# Etapa 3 — Reconhecimento de Entidades Nomeadas (NER)

**Objetivo desta etapa:** implementar e avaliar um modelo de Reconhecimento de Entidades Nomeadas (*Named Entity Recognition — NER*) para identificar automaticamente marcas, produtos, empresas e pessoas mencionadas em textos em português.

**Dados:** avaliações de produtos da base RePro e comentários sintéticos em português no estilo YouTube. Posteriormente, serão utilizados comentários reais coletados do YouTube para avaliar o desempenho do modelo.

**Modelo:** GLiNER2 PT-BR ([ottema/gliner2-ptbr](https://huggingface.co/ottema/gliner2-ptbr)), modelo pré-treinado disponibilizado no Hugging Face, voltado à extração de entidades em português brasileiro.

**Por que GLiNER2 e não spaCy?** O modelo `pt_core_news` do spaCy só reconhece as classes `PER`, `ORG`, `LOC` e `MISC`: não existe uma classe de **produto**, que é central para o monitoramento de marca. O GLiNER2 é um modelo de NER de **categorias livres** (*zero-shot*): as categorias são passadas como texto no momento da extração (por exemplo, `"marca"`, `"produto"`), sem precisar treinar o modelo de novo.

**Roteiro do notebook:**

1. Preparar o ambiente e carregar o modelo pré-treinado.
2. Testar o modelo em comentários simples.
3. Carregar a base RePro e selecionar uma amostra exploratória.
4. Aplicar o NER na amostra do RePro.
5. Comparar com categorias mais específicas.
6. Testar comentários no estilo YouTube.
7. Testar o parâmetro `threshold` (limiar de confiança).
8. Mini-avaliação quantitativa com um gabarito anotado à mão (precisão, revocação e F1).
9. Analisar os resultados e as limitações do modelo.

**Integração com a Etapa 2:** o modelo NER será utilizado em conjunto com o classificador de sentimentos desenvolvido anteriormente, permitindo identificar tanto as entidades mencionadas quanto os sentimentos expressos nos comentários.

## 1. Preparar o ambiente e carregar o modelo

As dependências estão no `requirements.txt` (`pip install -r src-pln/requirements.txt`). O notebook deve ser executado a partir da pasta `src-pln/notebooks/`, pois os caminhos são relativos (`../data/RePro.csv`).

Na primeira execução, o modelo (~1 GB) é baixado do Hugging Face e fica em cache.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from gliner2 import AutoExtractor

pd.set_option("display.max_colwidth", 200)
SEED = 42

print("torch:", torch.__version__)
print("Diretório atual:", Path.cwd())

C:\Users\pedro\OneDrive\Área de Trabalho\dev\PLN\social-listening-yt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.15.0+cpu
Diretório atual: C:\Users\pedro\OneDrive\Área de Trabalho\dev\PLN\social-listening-yt\src-pln\notebooks


In [2]:
extractor = AutoExtractor.from_pretrained("ottema/gliner2-ptbr")

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


C:\Users\pedro\OneDrive\Área de Trabalho\dev\PLN\social-listening-yt\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 2. Testar o modelo em comentários simples

Primeiro teste com quatro frases escritas por nós, usando as quatro categorias de interesse do projeto. O GLiNER2 devolve um dicionário `{"entities": {categoria: [trechos]}}`.

In [3]:
comentarios = [
    "Comprei um Samsung Galaxy S24 e achei a câmera incrível!",
    "Meu iPhone 15 da Apple começou a travar depois da atualização.",
    "A Natura tem produtos ótimos, mas o perfume Kaiak não dura muito.",
    "Prefiro os celulares da Motorola aos da Xiaomi.",
]

categorias = ["marca", "produto", "empresa", "pessoa"]

for comentario in comentarios:
    resultado = extractor.extract_entities(comentario, categorias)
    print(f"Comentário: {comentario}")
    print(f"Entidades:  {resultado['entities']}")
    print("-" * 60)

Comentário: Comprei um Samsung Galaxy S24 e achei a câmera incrível!
Entidades:  {'marca': ['Samsung'], 'produto': ['Samsung'], 'empresa': ['Samsung'], 'pessoa': []}
------------------------------------------------------------
Comentário: Meu iPhone 15 da Apple começou a travar depois da atualização.
Entidades:  {'marca': ['Apple'], 'produto': ['iPhone 15'], 'empresa': ['Apple'], 'pessoa': []}
------------------------------------------------------------
Comentário: A Natura tem produtos ótimos, mas o perfume Kaiak não dura muito.
Entidades:  {'marca': ['Natura', 'Kaiak'], 'produto': ['Natura'], 'empresa': ['Natura'], 'pessoa': ['Kaiak']}
------------------------------------------------------------
Comentário: Prefiro os celulares da Motorola aos da Xiaomi.
Entidades:  {'marca': ['Xiaomi', 'Motorola'], 'produto': ['celulares'], 'empresa': ['Motorola', 'Xiaomi'], 'pessoa': []}
------------------------------------------------------------


## 3. Carregar a base RePro (Americanas)

A RePro tem as colunas `product_brand` e `product_name`, que ajudam a conferir se a marca e o produto citados no texto foram reconhecidos. Ela **não** tem anotação de entidades, por isso aqui o uso é exploratório.

In [4]:
# O arquivo é baixado por `python utils/baixar_dados.py` (rodar a partir de src-pln/)
CAMINHO = Path("../data/RePro.csv")
df = pd.read_csv(CAMINHO)
print(df.shape)
df[["product_name", "product_brand", "review_title", "review_text", "overall_rating", "polarity"]].head()

(10003, 22)


,product_name,product_brand,review_title,review_text,overall_rating,polarity
0,Kit Com 10 Cuecas Boxer de Cotton 4.0 - Polo Match,NaN,"""O PRODUTO QUE NÃO ESTÁ AGRADANDO É A AMERICANAS.""","ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERICANAS JÁ NÃO ME INSPIRA MUITA CONFIANÇA. NÃO PELA AMERICANAS EM SI, POIS SEMPRE FORA UMA EMPRESA SÉRIA. A ÚLTIMA COMPRA QUE EFETUEI, A AMERICANAS COLOC...",3,"['NEGATIVO', 'POSITIVO']"
1,"Smartphone Motorola Moto G 5S Dual Chip Android 7.1.1 Nougat Tela 5.2"" Snapdragon 430 32GB 4G Câmera 16MP - Dourado",NaN,"""compra sem susto""","Aparelho muito bom, confiável e com valor aquisitivo em conta, ou seja: tudo que a massa precisa!",4,['POSITIVO']
2,Impressora Hp Sprocket 100 Jato de Tinta,NaN,*,"As cores da impressão não são fiéis, O custo benefício não vale a pena",3,['NEGATIVO']
3,Livro - It: A Coisa,NaN,***,"A história é muito boa, porém o autor ""enrolou"" um pouco no meio do livro.",3,"['NEGATIVO', 'POSITIVO']"
4,Termometro Laser Digital Infravermelho Febre De Testa Bebê,NaN,****,"Entrega rápida, produto muito bom Amei. Praticidade",4,['POSITIVO']


## 4. Aplicar o NER na amostra do RePro

Amostra aleatória de 10 avaliações (`random_state=42`), usada como **teste exploratório**. Uma só passada do modelo: imprime as entidades e monta a tabela.

In [5]:
amostra = df.dropna(subset=["review_text"]).sample(n=10, random_state=SEED)


def aplicar_ner(amostra, categorias, nomes_colunas, imprimir=True):
    # Roda o modelo uma vez por avaliação, imprime e devolve um DataFrame.
    linhas = []
    for _, linha in amostra.iterrows():
        comentario = str(linha["review_text"])
        entidades = extractor.extract_entities(comentario, categorias)["entities"]
        if imprimir:
            print("Comentário:", comentario)
            print("Marca cadastrada:", linha["product_brand"], "| Produto cadastrado:", linha["product_name"])
            print("Entidades:", entidades)
            print("-" * 60)
        registro = {
            "comentario": comentario,
            "marca_cadastrada": linha["product_brand"],
            "produto_cadastrado": linha["product_name"],
        }
        for cat, nome in zip(categorias, nomes_colunas):
            registro[nome] = entidades.get(cat, [])
        linhas.append(registro)
    return pd.DataFrame(linhas)


colunas = ["marcas", "produtos", "empresas", "pessoas"]
df_ner = aplicar_ner(amostra, categorias, colunas)
df_ner

Comentário: Diferente da foto, nem se parece um tapete, me decepcionei.
Marca cadastrada: corttex casa | Produto cadastrado: Tapete Jumper Chevron Blue Retangular (150x200cm) - Corttex Casa
Entidades: {'marca': [], 'produto': ['tapete'], 'empresa': [], 'pessoa': []}
------------------------------------------------------------
Comentário: Produto de alta qualidade, som e imagem perfeitos. Entrega rápida!
Marca cadastrada: nan | Produto cadastrado: Smart TV LED 32'' Semp Toshiba TCL 32L2600 HD com Conversor Digital 3 HDMI 2 USB Wi-Fi
Entidades: {'marca': [], 'produto': ['som'], 'empresa': [], 'pessoa': []}
------------------------------------------------------------


Comentário: muito bem feito, ele é todo de borracha, e tecido por dentro, tem furos respirador do pé. Muito bem feito eu super recomendo.
Marca cadastrada: nan | Produto cadastrado: Tênis Infantil Masculino Homem Aranha Grendene Kids 21361
Entidades: {'marca': ['borracha'], 'produto': [], 'empresa': [], 'pessoa': []}
------------------------------------------------------------


Comentário: Comprei esse patins para minha filha justamente pq é regulável, porém qdo precisei não consegui, ele é todo rebitado por dentro, não recomendo esse produto
Marca cadastrada: nan | Produto cadastrado: Patins Infantil 4 Rodas Quad Ajustável Winmax Tamanho M (33 a 36) WME05817AM Ahead Sports
Entidades: {'marca': [], 'produto': ['patins'], 'empresa': [], 'pessoa': ['filha']}
------------------------------------------------------------
Comentário: Não posso avaliar um produto que comprei no dia 18/12 e até hoje 17/01 vocês não enviaram!! todo dia entro na minha conta e o status é sempre o msm "aguardando nota fiscal"
Marca cadastrada: nan | Produto cadastrado: Tênis Fly Rainha
Entidades: {'marca': ['msm'], 'produto': ['msm'], 'empresa': ['msm'], 'pessoa': []}
------------------------------------------------------------


Comentário: O pagamento foi aprovado em 13/03/2018 e hoje são 11/05 e ainda não recebi o produto!!!! gostaria de explicações!!!!!!!
Marca cadastrada: nan | Produto cadastrado: Mesa De Centro Com 1 Prateleira Caribe Artely Cappuccino/preto
Entidades: {'marca': [], 'produto': [], 'empresa': [], 'pessoa': []}
------------------------------------------------------------


Comentário: Otimo produto, site totalmente confiavel, recomendo para outras pessoas...
Marca cadastrada: lg | Produto cadastrado: Smartphone Lg K10 Power Dual Chip Android Tela 5,5" Octacore Android 7.0 Nougat 32GB 4G Wi-Fi Câmera 13MP - Dourado
Entidades: {'marca': [], 'produto': [], 'empresa': [], 'pessoa': []}
------------------------------------------------------------
Comentário: Comprei a está pipoqueira com o polegar do Michey e chegou a outra de bolinha. E estou desde10/01tentando resolver esta situação.
Marca cadastrada: nan | Produto cadastrado: Pipoqueira Elétrica Mallory Disney Mickey
Entidades: {'marca': ['bolinha'], 'produto': ['pipoqueira'], 'empresa': [], 'pessoa': ['Michey']}
------------------------------------------------------------


Comentário: Infelizmente, recebi outro produto, diferente do que havia sido pedido (cadeira secretária gás branca corino roxo). Recebi uma cadeira toda preta e não consigo fazer contato para solicitar a troca.
Marca cadastrada: classic home | Produto cadastrado: Cadeira Secretária a Gás Branca Corino Roxo - Classic Home
Entidades: {'marca': ['preta'], 'produto': [], 'empresa': [], 'pessoa': []}
------------------------------------------------------------


Comentário: Até o momento não recebi o produto, não consigo reclamar com ninguém.
Marca cadastrada: nan | Produto cadastrado: Prateleira Decorativa Bpl 26 Linha Nature - Brv Móveis
Entidades: {'marca': [], 'produto': ['produto'], 'empresa': [], 'pessoa': []}
------------------------------------------------------------


,comentario,marca_cadastrada,produto_cadastrado,marcas,produtos,empresas,pessoas
0,"Diferente da foto, nem se parece um tapete, me decepcionei.",corttex casa,Tapete Jumper Chevron Blue Retangular (150x200cm) - Corttex Casa,[],[tapete],[],[]
1,"Produto de alta qualidade, som e imagem perfeitos. Entrega rápida!",NaN,Smart TV LED 32'' Semp Toshiba TCL 32L2600 HD com Conversor Digital 3 HDMI 2 USB Wi-Fi,[],[som],[],[]
2,"muito bem feito, ele é todo de borracha, e tecido por dentro, tem furos respirador do pé. Muito bem feito eu super recomendo.",NaN,Tênis Infantil Masculino Homem Aranha Grendene Kids 21361,[borracha],[],[],[]
3,"Comprei esse patins para minha filha justamente pq é regulável, porém qdo precisei não consegui, ele é todo rebitado por dentro, não recomendo esse produto",NaN,Patins Infantil 4 Rodas Quad Ajustável Winmax Tamanho M (33 a 36) WME05817AM Ahead Sports,[],[patins],[],[filha]
4,"Não posso avaliar um produto que comprei no dia 18/12 e até hoje 17/01 vocês não enviaram!! todo dia entro na minha conta e o status é sempre o msm ""aguardando nota fiscal""",NaN,Tênis Fly Rainha,[msm],[msm],[msm],[]
5,O pagamento foi aprovado em 13/03/2018 e hoje são 11/05 e ainda não recebi o produto!!!! gostaria de explicações!!!!!!!,NaN,Mesa De Centro Com 1 Prateleira Caribe Artely Cappuccino/preto,[],[],[],[]
6,"Otimo produto, site totalmente confiavel, recomendo para outras pessoas...",lg,"Smartphone Lg K10 Power Dual Chip Android Tela 5,5"" Octacore Android 7.0 Nougat 32GB 4G Wi-Fi Câmera 13MP - Dourado",[],[],[],[]
7,Comprei a está pipoqueira com o polegar do Michey e chegou a outra de bolinha. E estou desde10/01tentando resolver esta situação.,NaN,Pipoqueira Elétrica Mallory Disney Mickey,[bolinha],[pipoqueira],[],[Michey]
8,"Infelizmente, recebi outro produto, diferente do que havia sido pedido (cadeira secretária gás branca corino roxo). Recebi uma cadeira toda preta e não consigo fazer contato para solicitar a troca.",classic home,Cadeira Secretária a Gás Branca Corino Roxo - Classic Home,[preta],[],[],[]
9,"Até o momento não recebi o produto, não consigo reclamar com ninguém.",NaN,Prateleira Decorativa Bpl 26 Linha Nature - Brv Móveis,[],[produto],[],[]


## 5. Comparação com categorias mais específicas

Como o GLiNER2 recebe as categorias em texto livre, o nome da categoria funciona como uma "instrução". Testamos nomes mais específicos (`"marca comercial"`, `"nome de produto"`...) na **mesma amostra**, para ver se diminuem os falsos positivos com palavras comuns.

In [6]:
categorias_novas = ["marca comercial", "nome de produto", "nome de empresa", "nome de pessoa"]

df_ner_especifico = aplicar_ner(amostra, categorias_novas, colunas)
df_ner_especifico

Comentário: Diferente da foto, nem se parece um tapete, me decepcionei.
Marca cadastrada: corttex casa | Produto cadastrado: Tapete Jumper Chevron Blue Retangular (150x200cm) - Corttex Casa
Entidades: {'marca comercial': [], 'nome de produto': ['tapete'], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: Produto de alta qualidade, som e imagem perfeitos. Entrega rápida!
Marca cadastrada: nan | Produto cadastrado: Smart TV LED 32'' Semp Toshiba TCL 32L2600 HD com Conversor Digital 3 HDMI 2 USB Wi-Fi
Entidades: {'marca comercial': [], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: muito bem feito, ele é todo de borracha, e tecido por dentro, tem furos respirador do pé. Muito bem feito eu super recomendo.
Marca cadastrada: nan | Produto cadastrado: Tênis Infantil Masculino Homem Aranha Grendene Kids 21361
Entidades: {'marca comercial': ['borracha'], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: Comprei esse patins para minha filha justamente pq é regulável, porém qdo precisei não consegui, ele é todo rebitado por dentro, não recomendo esse produto
Marca cadastrada: nan | Produto cadastrado: Patins Infantil 4 Rodas Quad Ajustável Winmax Tamanho M (33 a 36) WME05817AM Ahead Sports
Entidades: {'marca comercial': [], 'nome de produto': ['patins'], 'nome de empresa': [], 'nome de pessoa': ['filha']}
------------------------------------------------------------


Comentário: Não posso avaliar um produto que comprei no dia 18/12 e até hoje 17/01 vocês não enviaram!! todo dia entro na minha conta e o status é sempre o msm "aguardando nota fiscal"
Marca cadastrada: nan | Produto cadastrado: Tênis Fly Rainha
Entidades: {'marca comercial': [], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: O pagamento foi aprovado em 13/03/2018 e hoje são 11/05 e ainda não recebi o produto!!!! gostaria de explicações!!!!!!!
Marca cadastrada: nan | Produto cadastrado: Mesa De Centro Com 1 Prateleira Caribe Artely Cappuccino/preto
Entidades: {'marca comercial': [], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: Otimo produto, site totalmente confiavel, recomendo para outras pessoas...
Marca cadastrada: lg | Produto cadastrado: Smartphone Lg K10 Power Dual Chip Android Tela 5,5" Octacore Android 7.0 Nougat 32GB 4G Wi-Fi Câmera 13MP - Dourado
Entidades: {'marca comercial': [], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: Comprei a está pipoqueira com o polegar do Michey e chegou a outra de bolinha. E estou desde10/01tentando resolver esta situação.
Marca cadastrada: nan | Produto cadastrado: Pipoqueira Elétrica Mallory Disney Mickey
Entidades: {'marca comercial': ['bolinha'], 'nome de produto': ['pipoqueira'], 'nome de empresa': [], 'nome de pessoa': ['Michey']}
------------------------------------------------------------


Comentário: Infelizmente, recebi outro produto, diferente do que havia sido pedido (cadeira secretária gás branca corino roxo). Recebi uma cadeira toda preta e não consigo fazer contato para solicitar a troca.
Marca cadastrada: classic home | Produto cadastrado: Cadeira Secretária a Gás Branca Corino Roxo - Classic Home
Entidades: {'marca comercial': ['preta'], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


Comentário: Até o momento não recebi o produto, não consigo reclamar com ninguém.
Marca cadastrada: nan | Produto cadastrado: Prateleira Decorativa Bpl 26 Linha Nature - Brv Móveis
Entidades: {'marca comercial': [], 'nome de produto': [], 'nome de empresa': [], 'nome de pessoa': []}
------------------------------------------------------------


,comentario,marca_cadastrada,produto_cadastrado,marcas,produtos,empresas,pessoas
0,"Diferente da foto, nem se parece um tapete, me decepcionei.",corttex casa,Tapete Jumper Chevron Blue Retangular (150x200cm) - Corttex Casa,[],[tapete],[],[]
1,"Produto de alta qualidade, som e imagem perfeitos. Entrega rápida!",NaN,Smart TV LED 32'' Semp Toshiba TCL 32L2600 HD com Conversor Digital 3 HDMI 2 USB Wi-Fi,[],[],[],[]
2,"muito bem feito, ele é todo de borracha, e tecido por dentro, tem furos respirador do pé. Muito bem feito eu super recomendo.",NaN,Tênis Infantil Masculino Homem Aranha Grendene Kids 21361,[borracha],[],[],[]
3,"Comprei esse patins para minha filha justamente pq é regulável, porém qdo precisei não consegui, ele é todo rebitado por dentro, não recomendo esse produto",NaN,Patins Infantil 4 Rodas Quad Ajustável Winmax Tamanho M (33 a 36) WME05817AM Ahead Sports,[],[patins],[],[filha]
4,"Não posso avaliar um produto que comprei no dia 18/12 e até hoje 17/01 vocês não enviaram!! todo dia entro na minha conta e o status é sempre o msm ""aguardando nota fiscal""",NaN,Tênis Fly Rainha,[],[],[],[]
5,O pagamento foi aprovado em 13/03/2018 e hoje são 11/05 e ainda não recebi o produto!!!! gostaria de explicações!!!!!!!,NaN,Mesa De Centro Com 1 Prateleira Caribe Artely Cappuccino/preto,[],[],[],[]
6,"Otimo produto, site totalmente confiavel, recomendo para outras pessoas...",lg,"Smartphone Lg K10 Power Dual Chip Android Tela 5,5"" Octacore Android 7.0 Nougat 32GB 4G Wi-Fi Câmera 13MP - Dourado",[],[],[],[]
7,Comprei a está pipoqueira com o polegar do Michey e chegou a outra de bolinha. E estou desde10/01tentando resolver esta situação.,NaN,Pipoqueira Elétrica Mallory Disney Mickey,[bolinha],[pipoqueira],[],[Michey]
8,"Infelizmente, recebi outro produto, diferente do que havia sido pedido (cadeira secretária gás branca corino roxo). Recebi uma cadeira toda preta e não consigo fazer contato para solicitar a troca.",classic home,Cadeira Secretária a Gás Branca Corino Roxo - Classic Home,[preta],[],[],[]
9,"Até o momento não recebi o produto, não consigo reclamar com ninguém.",NaN,Prateleira Decorativa Bpl 26 Linha Nature - Brv Móveis,[],[],[],[]


## 6. Testes em comentários no estilo do YouTube

Frases sintéticas que imitam o domínio final do projeto (gírias, tom informal). Usamos as categorias específicas da seção 5.

In [7]:
comentarios_youtube = [
    "Mano, esse Galaxy S24 da Samsung tá absurdo de bom!",
    "Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.",
    "Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?",
    "A câmera da Motorola tá muito melhor que a da Samsung.",
    "Comprei um perfume Kaiak da Natura e gostei demais!",
    "Esse vídeo do Felipe Neto falando da Netflix ficou muito bom.",
]

resultados_youtube = []
for comentario in comentarios_youtube:
    entidades = extractor.extract_entities(comentario, categorias_novas)["entities"]
    registro = {"comentario": comentario}
    for cat, nome in zip(categorias_novas, colunas):
        registro[nome] = entidades.get(cat, [])
    resultados_youtube.append(registro)

df_youtube = pd.DataFrame(resultados_youtube)
df_youtube

,comentario,marcas,produtos,empresas,pessoas
0,"Mano, esse Galaxy S24 da Samsung tá absurdo de bom!",[Samsung],[Galaxy S24],[Samsung],[Mano]
1,"Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.",[Apple],"[iPhone 15, Apple]",[Apple],[]
2,Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?,[Xiaomi],[Redmi Note 13],[Xiaomi],[]
3,A câmera da Motorola tá muito melhor que a da Samsung.,"[Motorola, Samsung]",[câmera],"[Motorola, Samsung]",[]
4,Comprei um perfume Kaiak da Natura e gostei demais!,"[Natura, Kaiak]","[Natura, perfume Kaiak]",[Natura],[]
5,Esse vídeo do Felipe Neto falando da Netflix ficou muito bom.,[Netflix],[],[Netflix],[Felipe Neto]


## 7. Teste do parâmetro `threshold`

O método `extract_entities` do GLiNER2 aceita o parâmetro `threshold` (limiar de confiança, padrão 0,5): só são devolvidas as entidades com confiança acima dele. Com `include_confidence=True`, o modelo mostra a confiança de cada entidade. A ideia é ver se um limiar mais alto elimina falsos positivos como **"Mano"** classificado como pessoa, sem perder as entidades corretas.

In [8]:
frase = comentarios_youtube[0]
print(frase)
print(extractor.extract_entities(frase, categorias_novas, include_confidence=True)["entities"])

Mano, esse Galaxy S24 da Samsung tá absurdo de bom!
{'marca comercial': [{'text': 'Samsung', 'confidence': 1.0}], 'nome de produto': [{'text': 'Galaxy S24', 'confidence': 1.0}], 'nome de empresa': [{'text': 'Samsung', 'confidence': 1.0}], 'nome de pessoa': [{'text': 'Mano', 'confidence': 0.9999587535858154}]}


In [9]:
limiares = [0.5, 0.9, 0.99, 0.999]
curto = dict(zip(categorias_novas, categorias))  # "marca comercial" -> "marca", ...
linhas = []
for comentario in comentarios_youtube:
    registro = {"comentario": comentario}
    for t in limiares:
        ents = extractor.extract_entities(comentario, categorias_novas, threshold=t)["entities"]
        registro[f"t={t}"] = "; ".join(f"{e} ({curto[c]})" for c, lista in ents.items() for e in lista)
    linhas.append(registro)

pd.DataFrame(linhas)

,comentario,t=0.5,t=0.9,t=0.99,t=0.999
0,"Mano, esse Galaxy S24 da Samsung tá absurdo de bom!",Samsung (marca); Galaxy S24 (produto); Samsung (empresa); Mano (pessoa),Samsung (marca); Galaxy S24 (produto); Samsung (empresa); Mano (pessoa),Samsung (marca); Galaxy S24 (produto); Samsung (empresa); Mano (pessoa),Samsung (marca); Galaxy S24 (produto); Samsung (empresa); Mano (pessoa)
1,"Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.",Apple (marca); iPhone 15 (produto); Apple (produto); Apple (empresa),Apple (marca); iPhone 15 (produto); Apple (produto); Apple (empresa),Apple (marca); iPhone 15 (produto); Apple (empresa),Apple (marca); iPhone 15 (produto); Apple (empresa)
2,Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?,Xiaomi (marca); Redmi Note 13 (produto); Xiaomi (empresa),Xiaomi (marca); Redmi Note 13 (produto); Xiaomi (empresa),Xiaomi (marca); Redmi Note 13 (produto); Xiaomi (empresa),Xiaomi (marca); Redmi Note 13 (produto); Xiaomi (empresa)
3,A câmera da Motorola tá muito melhor que a da Samsung.,Motorola (marca); Samsung (marca); câmera (produto); Motorola (empresa); Samsung (empresa),Motorola (marca); Samsung (marca); câmera (produto); Motorola (empresa); Samsung (empresa),Motorola (marca); Samsung (marca); câmera (produto); Motorola (empresa),Motorola (marca); Samsung (marca); câmera (produto); Motorola (empresa)
4,Comprei um perfume Kaiak da Natura e gostei demais!,Natura (marca); Kaiak (marca); Natura (produto); perfume Kaiak (produto); Natura (empresa),Natura (marca); Kaiak (marca); Natura (produto); perfume Kaiak (produto); Natura (empresa),Natura (marca); Kaiak (marca); Natura (produto); Natura (empresa),Natura (marca); Kaiak (marca); Natura (produto); Natura (empresa)
5,Esse vídeo do Felipe Neto falando da Netflix ficou muito bom.,Netflix (marca); Netflix (empresa); Felipe Neto (pessoa),Netflix (marca); Netflix (empresa); Felipe Neto (pessoa),Netflix (marca); Netflix (empresa); Felipe Neto (pessoa),Netflix (marca); Netflix (empresa); Felipe Neto (pessoa)


## 8. Mini-avaliação quantitativa (gabarito sintético)

> **Atenção:** esta é uma avaliação **pequena e sintética**. Os comentários abaixo foram escritos por nós, no estilo YouTube, e o gabarito (*gold*) foi anotado à mão. Ela serve para ter um primeiro número de precisão, revocação e F1 e para comparar configurações do modelo, **não** para medir o desempenho real. A avaliação definitiva será feita depois, com comentários reais do YouTube rotulados pelo Pedro.

### Critérios de anotação

- **marca:** nome de fabricante ou marca citada como dona de produtos (Samsung, Apple, Xiaomi, Natura, JBL...).
- **produto:** nome ou apelido de um produto/modelo específico (Galaxy S24, iPhone 15, Kaiak, "zap", "redmi"). Quando a marca vem colada no modelo ("Samsung Galaxy S24", "jbl tune 510"), anotamos marca e produto separados.
- **empresa:** empresa citada como organização ou serviço, não como marca de um produto (Netflix, Mercado Livre, Nubank, iFood...).
- **pessoa:** nome ou apelido de pessoa (Felipe Neto, Casimiro...). Gírias de tratamento como "mano" e "parça" **não** são pessoas.
- Palavras genéricas (câmera, celular, perfume, air fryer) **não** são entidades.

A fronteira entre **marca** e **empresa** é ambígua (a Samsung é as duas coisas), por isso também calculamos uma métrica que ignora a categoria.

O conjunto tem os 10 comentários das seções 2 e 6 e mais 14 frases novas com gírias, letras minúsculas e apelidos de produtos.

In [10]:
gold = [
    # Seção 2
    ("Comprei um Samsung Galaxy S24 e achei a câmera incrível!", [("Samsung", "marca"), ("Galaxy S24", "produto")]),
    ("Meu iPhone 15 da Apple começou a travar depois da atualização.", [("iPhone 15", "produto"), ("Apple", "marca")]),
    ("A Natura tem produtos ótimos, mas o perfume Kaiak não dura muito.", [("Natura", "marca"), ("Kaiak", "produto")]),
    ("Prefiro os celulares da Motorola aos da Xiaomi.", [("Motorola", "marca"), ("Xiaomi", "marca")]),
    # Seção 6
    ("Mano, esse Galaxy S24 da Samsung tá absurdo de bom!", [("Galaxy S24", "produto"), ("Samsung", "marca")]),
    ("Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.", [("iPhone 15", "produto"), ("Apple", "marca")]),
    ("Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?", [("Redmi Note 13", "produto"), ("Xiaomi", "marca")]),
    ("A câmera da Motorola tá muito melhor que a da Samsung.", [("Motorola", "marca"), ("Samsung", "marca")]),
    ("Comprei um perfume Kaiak da Natura e gostei demais!", [("Kaiak", "produto"), ("Natura", "marca")]),
    ("Esse vídeo do Felipe Neto falando da Netflix ficou muito bom.", [("Felipe Neto", "pessoa"), ("Netflix", "empresa")]),
    # Frases novas
    ("kkkkk o zap travou dnv, ja era", [("zap", "produto")]),
    ("troquei meu redmi por um iphone 13 e me arrependi kkk", [("redmi", "produto"), ("iphone 13", "produto")]),
    ("o s23 ultra é bom mas a samsung cobra caro demais nas peças", [("s23 ultra", "produto"), ("samsung", "marca")]),
    ("comprei no mercado livre e chegou em 2 dias, top demais", [("mercado livre", "empresa")]),
    ("alguém mais acha que o casimiro devia fazer review do ps5?", [("casimiro", "pessoa"), ("ps5", "produto")]),
    ("o nubank bloqueou meu cartão do nada, que raiva", [("nubank", "empresa")]),
    ("a air fryer da mondial pifou em 3 meses, nunca mais", [("mondial", "marca")]),
    ("o whindersson falou da shopee no vídeo e eu fui lá comprar kkk", [("whindersson", "pessoa"), ("shopee", "empresa")]),
    ("melhor fone que já tive foi o jbl tune 510, grave absurdo", [("jbl", "marca"), ("tune 510", "produto")]),
    ("a magalu atrasou minha entrega de novo, péssimo atendimento", [("magalu", "empresa")]),
    ("meu note da lenovo esquenta muito jogando valorant", [("lenovo", "marca"), ("valorant", "produto")]),
    ("o ifood ta cobrando taxa absurda, vou voltar pro rappi", [("ifood", "empresa"), ("rappi", "empresa")]),
    ("Alanzoka jogando no pc gamer da Positivo foi a melhor live do ano", [("Alanzoka", "pessoa"), ("Positivo", "marca")]),
    ("parça, a coca zero mudou o gosto ou é impressão minha?", [("coca zero", "produto")]),
]

CATEGORIAS_AVAL = ["marca", "produto", "empresa", "pessoa"]
print(f"{len(gold)} comentários, {sum(len(g) for _, g in gold)} entidades no gabarito")
pd.Series([c for _, g in gold for _, c in g]).value_counts()

24 comentários, 42 entidades no gabarito


marca      16
produto    15
empresa     7
pessoa      4
Name: count, dtype: int64

### Métricas

Comparamos o conjunto de pares `(texto, categoria)` previsto com o do gabarito, comentário por comentário:

- **Estrita:** acerto só quando o texto (sem diferença de maiúsculas/pontuação nas pontas) **e** a categoria batem exatamente.
- **Ignorando a categoria:** acerto quando o texto bate, qualquer que seja a categoria. Mostra quanto do erro vem da confusão marca/empresa/produto.

Precisão = acertos / previstos; revocação = acertos / gabarito; F1 = média harmônica das duas. O "total" é *micro* (soma os acertos de todas as categorias).

In [11]:
def normalizar(texto):
    return re.sub(r"^\W+|\W+$", "", texto.lower().strip())


def prever(textos, categorias, mapa, threshold=0.5, uma_categoria=False):
    # mapa: categoria usada no modelo -> categoria do gabarito
    # uma_categoria=True: se o mesmo trecho vier em várias categorias, fica só a de maior
    # confiança (empate: a que aparece primeiro na lista de categorias)
    previsoes = []
    for texto in textos:
        ents = extractor.extract_entities(texto, categorias, threshold=threshold,
                                          include_confidence=True)["entities"]
        pares = [(normalizar(e["text"]), mapa[c], e["confidence"], categorias.index(c))
                 for c, lista in ents.items() for e in lista]
        if uma_categoria:
            melhor = {}
            for trecho, cat, conf, ordem in pares:
                if trecho not in melhor or (-conf, ordem) < (-melhor[trecho][1], melhor[trecho][2]):
                    melhor[trecho] = (cat, conf, ordem)
            previsoes.append({(t, v[0]) for t, v in melhor.items()})
        else:
            previsoes.append({(t, c) for t, c, _, _ in pares})
    return previsoes


def prf(tp, n_prev, n_gold):
    p = tp / n_prev if n_prev else 0.0
    r = tp / n_gold if n_gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def avaliar(previsoes, gold):
    golds = [{(normalizar(e), c) for e, c in g} for _, g in gold]
    linhas = []
    for cat in CATEGORIAS_AVAL + ["TOTAL"]:
        sel = (lambda c: True) if cat == "TOTAL" else (lambda c, cat=cat: c == cat)
        tp = n_prev = n_gold = 0
        for prev, g in zip(previsoes, golds):
            prev_c = {x for x in prev if sel(x[1])}
            gold_c = {x for x in g if sel(x[1])}
            tp += len(prev_c & gold_c)
            n_prev += len(prev_c)
            n_gold += len(gold_c)
        p, r, f = prf(tp, n_prev, n_gold)
        linhas.append({"categoria": cat, "gabarito": n_gold, "previstas": n_prev, "acertos": tp,
                       "precisão": p, "revocação": r, "F1": f})
    # Correspondência que ignora a categoria (só no total)
    tp = sum(len({e for e, _ in prev} & {e for e, _ in g}) for prev, g in zip(previsoes, golds))
    n_prev = sum(len({e for e, _ in prev}) for prev in previsoes)
    n_gold = sum(len({e for e, _ in g}) for g in golds)
    p, r, f = prf(tp, n_prev, n_gold)
    linhas.append({"categoria": "TOTAL (ignora categoria)", "gabarito": n_gold, "previstas": n_prev, "acertos": tp,
                   "precisão": p, "revocação": r, "F1": f})
    return pd.DataFrame(linhas).set_index("categoria").round(3)


textos_gold = [t for t, _ in gold]
mapa_simples = {c: c for c in categorias}
mapa_especifico = dict(zip(categorias_novas, categorias))

prev_simples = prever(textos_gold, categorias, mapa_simples)
prev_especifico = prever(textos_gold, categorias_novas, mapa_especifico)

print("Categorias simples (marca, produto, empresa, pessoa) — threshold 0,5")
display(avaliar(prev_simples, gold))
print("Categorias específicas (marca comercial, nome de produto, ...) — threshold 0,5")
display(avaliar(prev_especifico, gold))

Categorias simples (marca, produto, empresa, pessoa) — threshold 0,5


,gabarito,previstas,acertos,precisão,revocação,F1
categoria,,,,,,
marca,16,26,16,0.615,1.000,0.762
produto,15,24,10,0.417,0.667,0.513
empresa,7,25,7,0.280,1.000,0.438
pessoa,4,7,4,0.571,1.000,0.727
TOTAL,42,82,37,0.451,0.881,0.597
TOTAL (ignora categoria),42,49,39,0.796,0.929,0.857


Categorias específicas (marca comercial, nome de produto, ...) — threshold 0,5


,gabarito,previstas,acertos,precisão,revocação,F1
categoria,,,,,,
marca,16,27,16,0.593,1.000,0.744
produto,15,27,11,0.407,0.733,0.524
empresa,7,26,7,0.269,1.000,0.424
pessoa,4,6,4,0.667,1.000,0.800
TOTAL,42,86,38,0.442,0.905,0.594
TOTAL (ignora categoria),42,49,40,0.816,0.952,0.879


### Efeito do `threshold` nas métricas

Repetimos a avaliação (F1 estrito total) variando o limiar, para os dois conjuntos de categorias.

In [12]:
linhas = []
for nome, cats, mapa in [("simples", categorias, mapa_simples), ("específicas", categorias_novas, mapa_especifico)]:
    for t in [0.3, 0.5, 0.7, 0.9, 0.99, 0.999]:
        tab = avaliar(prever(textos_gold, cats, mapa, threshold=t), gold)
        linhas.append({"categorias": nome, "threshold": t,
                       "precisão": tab.loc["TOTAL", "precisão"], "revocação": tab.loc["TOTAL", "revocação"],
                       "F1": tab.loc["TOTAL", "F1"], "F1 (ignora categoria)": tab.loc["TOTAL (ignora categoria)", "F1"]})

pd.DataFrame(linhas).pivot(index="threshold", columns="categorias")

precisão           revocação                  F1          \
categorias específicas simples específicas simples específicas simples   
threshold                                                                
0.300            0.432   0.430       0.905   0.881       0.585   0.578   
0.500            0.442   0.451       0.905   0.881       0.594   0.597   
0.700            0.447   0.457       0.905   0.881       0.598   0.602   
0.900            0.462   0.481       0.881   0.881       0.607   0.622   
0.990            0.486   0.493       0.833   0.857       0.614   0.626   
0.999            0.516   0.537       0.786   0.857       0.623   0.661   

           F1 (ignora categoria)          
categorias           específicas simples  
threshold                                 
0.300                      0.879   0.848  
0.500                      0.879   0.857  
0.700                      0.879   0.857  
0.900                      0.879   0.857  
0.990                      0.864   0.844  
0.999                      0.857   0.864

### Uma categoria por trecho

O GLiNER2 costuma devolver o mesmo trecho em mais de uma categoria (por exemplo, "Samsung" como marca **e** como empresa), o que derruba a precisão estrita. Um pós-processamento simples é manter, para cada trecho, só a categoria de maior confiança (no empate, a primeira da lista: marca > produto > empresa > pessoa).

In [13]:
prev_uma = prever(textos_gold, categorias, mapa_simples, uma_categoria=True)
print("Categorias simples + uma categoria por trecho — threshold 0,5")
avaliar(prev_uma, gold)

Categorias simples + uma categoria por trecho — threshold 0,5


,gabarito,previstas,acertos,precisão,revocação,F1
categoria,,,,,,
marca,16,17,11,0.647,0.688,0.667
produto,15,16,9,0.562,0.600,0.581
empresa,7,10,5,0.500,0.714,0.588
pessoa,4,6,4,0.667,1.000,0.800
TOTAL,42,49,29,0.592,0.690,0.637
TOTAL (ignora categoria),42,49,39,0.796,0.929,0.857


### Análise de erros

Para a configuração padrão (categorias simples, threshold 0,5): falsos positivos (previstos que não estão no gabarito) e falsos negativos (do gabarito que o modelo não achou), comentário por comentário.

In [14]:
def tabela_erros(previsoes, gold):
    linhas = []
    for (texto, g), prev in zip(gold, previsoes):
        g = {(normalizar(e), c) for e, c in g}
        fp, fn = sorted(prev - g), sorted(g - prev)
        if fp or fn:
            linhas.append({"comentario": texto,
                           "falsos positivos": "; ".join(f"{e} ({c})" for e, c in fp),
                           "falsos negativos": "; ".join(f"{e} ({c})" for e, c in fn)})
    return pd.DataFrame(linhas)


erros = tabela_erros(prev_simples, gold)
print(f"Comentários com algum erro: {len(erros)} de {len(gold)}")
erros

Comentários com algum erro: 22 de 24


,comentario,falsos positivos,falsos negativos
0,Comprei um Samsung Galaxy S24 e achei a câmera incrível!,samsung (empresa); samsung (produto),galaxy s24 (produto)
1,Meu iPhone 15 da Apple começou a travar depois da atualização.,apple (empresa),
2,"A Natura tem produtos ótimos, mas o perfume Kaiak não dura muito.",kaiak (marca); kaiak (pessoa); natura (empresa); natura (produto),kaiak (produto)
3,Prefiro os celulares da Motorola aos da Xiaomi.,celulares (produto); motorola (empresa); xiaomi (empresa),
4,"Mano, esse Galaxy S24 da Samsung tá absurdo de bom!",mano (pessoa); samsung (empresa),
5,"Depois dessa atualização meu iPhone 15 ficou horrível, Apple vacilou.",apple (empresa); apple (produto),
6,Alguém sabe se o Redmi Note 13 da Xiaomi vale a pena?,xiaomi (empresa),
7,A câmera da Motorola tá muito melhor que a da Samsung.,câmera (produto); motorola (empresa); samsung (empresa),
8,Comprei um perfume Kaiak da Natura e gostei demais!,kaiak (marca); natura (empresa); natura (produto); perfume kaiak (produto),kaiak (produto)
9,Esse vídeo do Felipe Neto falando da Netflix ficou muito bom.,netflix (marca),


## 9. Análise dos resultados

### Metodologia

Usamos o GLiNER2 PT-BR (`ottema/gliner2-ptbr`) com dois conjuntos de categorias: simples (`marca`, `produto`, `empresa`, `pessoa`) e específicas (`marca comercial`, `nome de produto`, `nome de empresa`, `nome de pessoa`). O modelo foi aplicado a:

- uma amostra aleatória de 10 avaliações da RePro (teste exploratório, sem gabarito);
- 24 comentários sintéticos no estilo YouTube, com gabarito de 42 entidades anotado à mão (16 marcas, 15 produtos, 7 empresas e 4 pessoas).

### Amostra da RePro (exploratória)

- Quase nenhuma das 10 avaliações cita a marca ou o produto pelo nome, então há pouco para acertar. Nas três que têm marca cadastrada (Corttex Casa, LG e Classic Home), a marca não aparece no texto.
- Com as categorias simples, o modelo marca palavras comuns como entidades: "tapete", "patins" e "som" como produto, "borracha", "bolinha" e "preta" como marca, "filha" como pessoa e "msm" (abreviação de "mesmo") como marca, produto e empresa ao mesmo tempo.
- As categorias específicas eliminaram parte desses erros ("som", "msm", "produto"), mas não todos: "borracha", "bolinha" e "preta" continuam como marca comercial e "filha" como pessoa.
- Um acerto parcial: "Michey" (erro de digitação de *Mickey*) foi reconhecido como pessoa; no contexto do produto, seria mais bem descrito como personagem ou marca licenciada.

### Mini-avaliação (24 comentários sintéticos)

Categorias simples, `threshold` 0,5:

| Categoria | Gabarito | Previstas | Precisão | Revocação | F1 |
|---|---|---|---|---|---|
| marca | 16 | 26 | 0,615 | 1,000 | 0,762 |
| produto | 15 | 24 | 0,417 | 0,667 | 0,513 |
| empresa | 7 | 25 | 0,280 | 1,000 | 0,438 |
| pessoa | 4 | 7 | 0,571 | 1,000 | 0,727 |
| **Total (estrito)** | 42 | 82 | **0,451** | **0,881** | **0,597** |
| Total (ignora categoria) | 42 | 49 | 0,796 | 0,929 | 0,857 |

- **A revocação é alta e a precisão é baixa.** O modelo encontra quase todas as entidades (88% com a categoria certa, 93% considerando só o texto), mas prevê quase o dobro do necessário (82 previsões para 42 entidades).
- **A maior causa de erro é repetir o mesmo trecho em várias categorias.** "Samsung", "Apple", "Motorola" e "Xiaomi" aparecem como marca **e** empresa, e "Nubank", "iFood" e "Shopee" como empresa **e** marca. Por isso a precisão de `empresa` fica em 0,280. Quando ignoramos a categoria, a precisão sobe de 0,451 para 0,796.
- **Produto é a categoria mais difícil (F1 0,513).** O modelo não reconheceu "tune 510" (fone JBL) nem "valorant" (jogo). Em "Comprei um Samsung Galaxy S24", ele classificou "Samsung" como produto e não achou "Galaxy S24". Em "perfume Kaiak", recortou o trecho inteiro em vez de "Kaiak" e ainda classificou "Kaiak" como marca e como pessoa. Também marcou palavras genéricas como produto ("celulares", "câmera", "cartão", "air fryer", "note", "pc gamer").
- **Gírias e apelidos.** Foram reconhecidos apelidos e nomes em minúsculas, como "zap", "redmi", "s23 ultra", "ps5", "casimiro", "whindersson" e "coca zero". Por outro lado, "Mano" e "parça" viraram pessoa e "dnv" (abreviação de "de novo") virou marca.
- **Categorias específicas** quase não mudaram o resultado: F1 estrito de 0,594 (contra 0,597) e F1 sem categoria de 0,879 (contra 0,857).

### Efeito do `threshold`

A confiança do GLiNER2 fica quase sempre perto de 1,0, inclusive nos erros: "Mano" sai como pessoa com confiança de 0,99996. Por isso, aumentar o limiar quase não filtra nada até valores extremos. Com as categorias simples, o F1 estrito vai de 0,597 (limiar 0,5) para 0,661 (0,999): a precisão sobe de 0,451 para 0,537 e a revocação cai de 0,881 para 0,857. **"Mano" continua classificado como pessoa em todos os limiares testados.** Ou seja, o `threshold` não resolve os falsos positivos de gíria.

### Uma categoria por trecho

Manter só a categoria de maior confiança para cada trecho aumenta a precisão estrita (de 0,451 para 0,592) e o F1 estrito (de 0,597 para 0,637), mas a revocação cai (de 0,881 para 0,690). Como as confianças empatam em ~1,0, a escolha entre marca e empresa acaba sendo quase arbitrária. Isso confirma que a fronteira marca/empresa é o ponto fraco, tanto do modelo quanto da própria definição das categorias.

### Limitações

- O conjunto de avaliação é **pequeno (24 frases, 42 entidades) e sintético**, escrito e anotado por nós. Os números servem para comparar configurações, não como estimativa do desempenho em comentários reais.
- A distinção entre marca e empresa é ambígua e depende do critério de anotação adotado.
- A correspondência é exata no texto: um recorte um pouco diferente ("perfume Kaiak" × "Kaiak") conta como erro.
- A RePro não tem anotação de entidades, então a amostra de 10 avaliações foi analisada apenas qualitativamente.

### Conclusão e próximos passos

O GLiNER2 é bom em **encontrar** as menções a marcas, produtos, empresas e pessoas, inclusive em textos informais (revocação de 0,93 sem considerar a categoria), mas é fraco em **atribuir uma única categoria** e confunde gírias com nomes próprios. Para o monitoramento de marca, isso sugere:

1. Juntar `marca` e `empresa` em uma categoria só ("marca/empresa"), já que, para o cliente, as duas indicam a mesma coisa: quem está sendo citado.
2. Usar uma lista de palavras a ignorar (gírias como "mano", "parça", "dnv") e/ou uma lista de marcas e produtos conhecidos para validar as extrações.
3. Fazer a avaliação definitiva com comentários reais do YouTube rotulados pelo Pedro, calculando as mesmas métricas (precisão, revocação e F1 por categoria) com o código deste notebook.